# Task 3 — Consultas analíticas e dashboard

Este notebook usa o projeto Python da pasta `src/` para consultar o Athena e montar o dashboard.

## 1. Imports e configuração

In [ ]:
from pathlib import Path
import sys

# Ajuste automático para importar src/task3 quando o notebook está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
from task3.athena import athena_read, validate_tables
from task3.dashboard import build_dashboard


## 2. Validação das tabelas no Athena

In [ ]:
df_validation = validate_tables()
df_validation


## 3. Consulta exploratória em `dim_products`

In [ ]:
sql_dim_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = athena_read(sql_dim_products)
df_products.head(20)


## 4. Vendas totais por país

In [ ]:
sql_sales_by_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries
    ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_by_country = athena_read(sql_sales_by_country)
df_sales_by_country


## 5. Base detalhada para dashboard

In [ ]:
sql_detailed_sales = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products
    ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries
    ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates
    ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_sales_detail = athena_read(sql_detailed_sales)
df_sales_detail['full_date'] = pd.to_datetime(df_sales_detail['full_date'])
df_sales_detail.head()


## 6. Dashboard interativo

In [ ]:
build_dashboard(df_sales_detail)


## 7. Validação opcional da métrica `sales_amount`

In [ ]:
sql_sales_amount_validation = """
SELECT
    COUNT(*) AS invalid_rows
FROM fact_orders
WHERE ABS(sales_amount - quantity_ordered * price_each) > 0.01
"""

df_sales_amount_validation = athena_read(sql_sales_amount_validation)
df_sales_amount_validation
